In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.gold_reader import (
    read_gold_table,
)

print("Gold reader importado com sucesso.")

In [ ]:
import os

print(
    "AWS_ACCESS_KEY_ID configurada:",
    bool(os.getenv("AWS_ACCESS_KEY_ID"))
)

print(
    "AWS_SECRET_ACCESS_KEY configurada:",
    bool(os.getenv("AWS_SECRET_ACCESS_KEY"))
)

print(
    "AWS_REGION:",
    os.getenv("AWS_REGION")
)

In [ ]:
indicador_gold = read_gold_table(
    "indicador_alfabetizacao_municipio"
)

print(indicador_gold.shape)
print(indicador_gold.columns.tolist())

indicador_gold.head()

In [ ]:
import src.preprocessing.gold_features as gf

print(gf.__file__)
print(
    [
        nome
        for nome in dir(gf)
        if "municipal" in nome.lower()
        or "gold" in nome.lower()
    ]
)

In [ ]:
from src.preprocessing.gold_features import (
    build_municipal_gold_features,
)

print("Função importada com sucesso.")

In [ ]:
from src.preprocessing.gold_reader import read_gold_table

indicador_gold = read_gold_table(
    "indicador_alfabetizacao_municipio"
)

print("Gold carregada:", indicador_gold.shape)

In [ ]:
municipal_features = build_municipal_gold_features(
    indicador_gold
)

print("Shape:", municipal_features.shape)

print("\nColunas:")
print(municipal_features.columns.tolist())

print("\nDistribuição por rede:")
print(municipal_features["rede"].value_counts())

print("\nDuplicidades por município + rede:")
print(
    municipal_features[
        ["id_municipio", "rede"]
    ].duplicated().sum()
)

print("\nValores nulos:")
print(municipal_features.isna().sum())

municipal_features.head()

In [ ]:
desempenho_gold = read_gold_table(
    "desempenho_alunos_municipio"
)

print("Shape:", desempenho_gold.shape)

print("\nColunas:")
print(desempenho_gold.columns.tolist())

print("\nAnos:")
print(desempenho_gold["ano"].value_counts().sort_index())

print("\nRedes:")
print(desempenho_gold["rede"].value_counts())

desempenho_gold.head()

In [ ]:
from src.preprocessing.gold_features import (
    build_municipal_gold_features,
    build_performance_gold_features,
)

print("Funções Gold importadas com sucesso.")

In [ ]:
performance_features = build_performance_gold_features(
    desempenho_gold
)

print("Shape:", performance_features.shape)

print("\nColunas:")
print(performance_features.columns.tolist())

print("\nDistribuição por rede:")
print(performance_features["rede"].value_counts())

print("\nDuplicidades por município + rede:")
print(
    performance_features[
        ["id_municipio", "rede"]
    ].duplicated().sum()
)

print("\nValores nulos:")
print(performance_features.isna().sum())

performance_features.head()

In [ ]:
metas_gold = read_gold_table(
    "comparativo_metas_resultados"
)

print("Shape:", metas_gold.shape)

print("\nColunas:")
print(metas_gold.columns.tolist())

print("\nNíveis geográficos:")
print(metas_gold["nivel_geografico"].value_counts())

print("\nAnos:")
print(metas_gold["ano"].value_counts().sort_index())

In [ ]:
municipios_metas = metas_gold.loc[
    (metas_gold["nivel_geografico"] == "municipio")
    & (metas_gold["ano"] == 2023)
    & (metas_gold["ano_meta"] == 2024)
].copy()

print("Shape município/meta 2024:", municipios_metas.shape)

print("\nRedes:")
print(municipios_metas["rede"].value_counts())

print("\nNulos:")
print(
    municipios_metas[
        [
            "id_municipio",
            "taxa_alfabetizacao",
            "meta_alfabetizacao",
            "gap_para_meta",
            "sigla_uf",
            "idhm",
            "idhm_educacao",
            "idhm_renda",
            "idhm_longevidade",
        ]
    ].isna().sum()
)

In [ ]:
ufs_metas = metas_gold.loc[
    (metas_gold["nivel_geografico"] == "uf")
    & (metas_gold["ano"] == 2023)
    & (metas_gold["ano_meta"] == 2024)
].copy()

print("Shape UF/meta 2024:", ufs_metas.shape)

print("\nRedes:")
print(ufs_metas["rede"].value_counts())

print("\nUFs únicas:")
print(ufs_metas["sigla_uf"].nunique())

print("\nDuplicidades UF + rede:")
print(
    ufs_metas[
        ["sigla_uf", "rede"]
    ].duplicated().sum()
)

ufs_metas[
    [
        "sigla_uf",
        "sigla_uf_nome",
        "rede",
        "taxa_alfabetizacao",
        "meta_alfabetizacao",
        "gap_para_meta",
        "idhm",
        "idhm_educacao",
        "idhm_renda",
        "idhm_longevidade",
    ]
].head(10)

In [ ]:
from src.preprocessing.gold_features import (
    build_municipal_gold_features,
    build_performance_gold_features,
    build_municipal_target_features,
)

In [ ]:
target_features = build_municipal_target_features(
    metas_gold
)

print("Shape:", target_features.shape)

print("\nDistribuição por rede:")
print(target_features["rede"].value_counts())

print("\nDuplicidades:")
print(
    target_features[
        ["id_municipio", "rede"]
    ].duplicated().sum()
)

print("\nNulos:")
print(target_features.isna().sum())

target_features.head()

In [ ]:
gold_comparison = municipal_features.merge(
    target_features,
    on=["id_municipio", "rede"],
    how="inner",
    suffixes=("_indicador", "_metas"),
)

print("Registros comparáveis:", len(gold_comparison))

print("\nDiferença - taxa de alfabetização:")
taxa_diff = (
    gold_comparison["taxa_alfabetizacao_municipio_2023"]
    - gold_comparison["taxa_alfabetizacao_meta_base_2023"]
).abs()

print(taxa_diff.describe())


print("\nDiferença - meta 2024:")
meta_diff = (
    gold_comparison["meta_alfabetizacao_municipio_2024_indicador"]
    - gold_comparison["meta_alfabetizacao_municipio_2024_metas"]
).abs()

print(meta_diff.describe())


print("\nDiferença - gap para meta:")
gap_diff = (
    gold_comparison["gap_para_meta_municipio_2024_indicador"]
    - gold_comparison["gap_para_meta_municipio_2024_metas"]
).abs()

print(gap_diff.describe())

In [ ]:
print("INDICADOR")
print(
    [
        col for col in indicador_gold.columns
        if "uf" in col.lower()
    ]
)

print("\nDESEMPENHO")
print(
    [
        col for col in desempenho_gold.columns
        if "uf" in col.lower()
    ]
)

print("\nMETAS")
print(
    [
        col for col in metas_gold.columns
        if "uf" in col.lower()
    ]
)

In [ ]:
municipios_com_uf = metas_gold.loc[
    (metas_gold["nivel_geografico"] == "municipio")
    & metas_gold["sigla_uf"].notna(),
    [
        "id_municipio",
        "id_municipio_nome",
        "sigla_uf",
    ],
].drop_duplicates()

print(
    "Municípios com UF preenchida na Gold de metas:",
    len(municipios_com_uf),
)

municipios_com_uf.head()

In [ ]:
from src.preprocessing.gold_features import (
    add_uf_from_municipality_code,
)

municipal_with_uf = add_uf_from_municipality_code(
    municipal_features
)

print("Shape:", municipal_with_uf.shape)

print("\nUFs únicas:")
print(municipal_with_uf["sigla_uf"].nunique())

print("\nNulos em sigla_uf:")
print(municipal_with_uf["sigla_uf"].isna().sum())

print("\nDistribuição das UFs:")
print(
    municipal_with_uf["sigla_uf"]
    .value_counts()
    .sort_index()
)

municipal_with_uf[
    [
        "id_municipio",
        "id_municipio_nome",
        "sigla_uf",
        "rede",
    ]
].head(10)

In [ ]:
idhm_uf = metas_gold.loc[
    (metas_gold["nivel_geografico"] == "uf")
    & (metas_gold["ano"] == 2023)
    & (metas_gold["ano_meta"] == 2024)
    & (metas_gold["rede"].isin(["Estadual", "Municipal"])),
    [
        "sigla_uf",
        "sigla_uf_nome",
        "rede",
        "idhm",
        "idhm_educacao",
        "idhm_renda",
        "idhm_longevidade",
    ],
].copy()

print("Shape:", idhm_uf.shape)

print("\nUFs únicas:")
print(idhm_uf["sigla_uf"].nunique())

print("\nRedes:")
print(idhm_uf["rede"].value_counts())

print("\nNulos:")
print(
    idhm_uf[
        [
            "idhm",
            "idhm_educacao",
            "idhm_renda",
            "idhm_longevidade",
        ]
    ].isna().sum()
)

print("\nDuplicidades UF + rede:")
print(
    idhm_uf[
        ["sigla_uf", "rede"]
    ].duplicated().sum()
)

In [ ]:
ufs_municipios = set(
    municipal_with_uf["sigla_uf"].dropna().unique()
)

ufs_idhm = set(
    idhm_uf["sigla_uf"].dropna().unique()
)

print("UFs presentes nos municípios:")
print(sorted(ufs_municipios))

print("\nUFs presentes no IDHM:")
print(sorted(ufs_idhm))

print("\nUFs dos municípios sem IDHM:")
print(sorted(ufs_municipios - ufs_idhm))

print("\nUFs com IDHM sem municípios:")
print(sorted(ufs_idhm - ufs_municipios))

In [ ]:
idhm_consistency = (
    idhm_uf
    .groupby("sigla_uf")[
        [
            "idhm",
            "idhm_educacao",
            "idhm_renda",
            "idhm_longevidade",
        ]
    ]
    .nunique()
)

print("Máximo de valores distintos por UF:")
print(idhm_consistency.max())

print("\nUFs com alguma divergência entre redes:")
print(
    idhm_consistency[
        (idhm_consistency > 1).any(axis=1)
    ]
)

In [ ]:
from src.preprocessing.gold_features import (
    add_uf_from_municipality_code,
    build_uf_socioeconomic_features,
)

In [ ]:
socioeconomic_features = build_uf_socioeconomic_features(
    metas_gold
)

print("Shape:", socioeconomic_features.shape)

print("\nUFs únicas:")
print(socioeconomic_features["sigla_uf"].nunique())

print("\nDuplicidades:")
print(
    socioeconomic_features[
        ["sigla_uf"]
    ].duplicated().sum()
)

print("\nValores nulos:")
print(socioeconomic_features.isna().sum())

socioeconomic_features.head(10)

consolidar Indicador + Desempenho

In [ ]:
gold_educational_features = municipal_features.merge(
    performance_features.drop(
        columns=["id_municipio_nome"]
    ),
    on=["id_municipio", "rede"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Shape:", gold_educational_features.shape)

print("\nResultado do merge:")
print(
    gold_educational_features["_merge"]
    .value_counts()
)

print("\nCobertura das features de desempenho:")
coverage = (
    gold_educational_features["_merge"]
    .eq("both")
    .mean()
    * 100
)

print(f"{coverage:.2f}%")

print("\nAusências por rede:")
print(
    gold_educational_features.loc[
        gold_educational_features["_merge"] == "left_only",
        "rede",
    ].value_counts()
)

In [ ]:
missing_performance = gold_educational_features.loc[
    gold_educational_features["_merge"] == "left_only",
    [
        "id_municipio",
        "id_municipio_nome",
        "rede",
    ],
]

print("Total sem correspondência:")
print(len(missing_performance))

print("\nExemplos:")
display(
    missing_performance.head(20)
)

Indicadores + Desempenho + Metas

In [ ]:
target_features_selected = target_features[
    [
        "id_municipio",
        "rede",
        "meta_alfabetizacao_municipio_2024",
        "gap_para_meta_municipio_2024",
        "atingiu_meta_municipio_2024",
    ]
].copy()

In [ ]:
gold_features_with_targets = (
    gold_educational_features
    .drop(columns=["_merge"])
    .merge(
        target_features_selected,
        on=["id_municipio", "rede"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

print("Shape:", gold_features_with_targets.shape)

print("\nResultado do merge:")
print(
    gold_features_with_targets["_merge"]
    .value_counts()
)

coverage = (
    gold_features_with_targets["_merge"]
    .eq("both")
    .mean()
    * 100
)

print(
    f"\nCobertura das features de metas: "
    f"{coverage:.2f}%"
)

print("\nAusências por rede:")
print(
    gold_features_with_targets.loc[
        gold_features_with_targets["_merge"] == "left_only",
        "rede",
    ].value_counts()
)

Agora vamos acrescentar IDHM

In [ ]:
gold_features_with_uf = (
    gold_features_with_targets
    .drop(columns=["_merge"])
)

gold_features_with_uf = add_uf_from_municipality_code(
    gold_features_with_uf
)

print("Shape:", gold_features_with_uf.shape)

print("\nUFs únicas:")
print(gold_features_with_uf["sigla_uf"].nunique())

print("\nNulos em UF:")
print(gold_features_with_uf["sigla_uf"].isna().sum())

In [ ]:
gold_consolidated_test = gold_features_with_uf.merge(
    socioeconomic_features,
    on="sigla_uf",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Shape:", gold_consolidated_test.shape)

print("\nResultado do merge socioeconômico:")
print(
    gold_consolidated_test["_merge"]
    .value_counts()
)

coverage = (
    gold_consolidated_test["_merge"]
    .eq("both")
    .mean()
    * 100
)

print(
    f"\nCobertura socioeconômica: "
    f"{coverage:.2f}%"
)

print("\nNulos nos indicadores socioeconômicos:")
print(
    gold_consolidated_test[
        [
            "idhm",
            "idhm_educacao",
            "idhm_renda",
            "idhm_longevidade",
        ]
    ].isna().sum()
)

In [ ]:
redundancy_check = (
    municipal_features
    .merge(
        performance_features,
        on=["id_municipio", "rede"],
        how="inner",
        suffixes=("_indicador", "_desempenho"),
    )
)

print("Registros comparáveis:", len(redundancy_check))


print("\n=== TAXA DE ALFABETIZAÇÃO ===")

taxa_diff = (
    redundancy_check["taxa_alfabetizacao_municipio_2023"]
    - redundancy_check["taxa_alfabetizacao_desempenho_2023"]
).abs()

print(taxa_diff.describe())


print("\n=== PROFICIÊNCIA / MÉDIA PORTUGUÊS ===")

proficiencia_diff = (
    redundancy_check["media_portugues_municipio_2023"]
    - redundancy_check["proficiencia_media_ponderada_2023"]
).abs()

print(proficiencia_diff.describe())


print("\n=== CORRELAÇÃO ===")

print(
    redundancy_check[
        [
            "media_portugues_municipio_2023",
            "proficiencia_media_ponderada_2023",
        ]
    ].corr()
)

Teste final da Gold consolidada

In [ ]:
from src.preprocessing.gold_features import (
    build_consolidated_gold_features,
)

gold_features = build_consolidated_gold_features(
    indicador_gold=indicador_gold,
    desempenho_gold=desempenho_gold,
    metas_gold=metas_gold,
)

print("Shape:", gold_features.shape)

print("\nColunas:")
print(gold_features.columns.tolist())

print("\nDuplicidades município + rede:")
print(
    gold_features[
        ["id_municipio", "rede"]
    ].duplicated().sum()
)

print("\nNulos:")
print(
    gold_features.isna().sum()
)

gold_features.head()

In [ ]:
from src.preprocessing.silver_reader import read_silver_students

students_current = read_silver_students()

print("Shape:", students_current.shape)
print("Colunas:", students_current.columns.tolist())

In [ ]:
from src.preprocessing.build_modeling_dataset import (
    build_modeling_dataset_from_gold,
)

modeling_gold = build_modeling_dataset_from_gold(
    alunos_df=students_current,
    gold_features_df=gold_features,
)

print("Shape:", modeling_gold.shape)

print("\nDuplicidades:")
print(modeling_gold["id_aluno"].duplicated().sum())

print("\nRedes:")
print(modeling_gold["rede"].value_counts(dropna=False))

print("\nTarget:")
print(modeling_gold["alfabetizado"].value_counts(dropna=False))

print("\nDisponibilidade Gold:")
print(
    modeling_gold[
        "gold_historico_disponivel"
    ].value_counts(dropna=False)
)

print("\nColunas:")
print(modeling_gold.columns.tolist())

In [ ]:
output_path = (
    "../data/processed/"
    "modeling_dataset_2024_gold.parquet"
)

modeling_gold.to_parquet(
    output_path,
    index=False,
)

print(
    "Dataset salvo em:",
    output_path
)

print(
    "Shape salvo:",
    modeling_gold.shape
)

In [ ]:
modeling_gold_check = pd.read_parquet(
    "../data/processed/"
    "modeling_dataset_2024_gold.parquet"
)

print("Shape:", modeling_gold_check.shape)

print(
    "Duplicidades:",
    modeling_gold_check[
        "id_aluno"
    ].duplicated().sum()
)

print(
    "Target nulo:",
    modeling_gold_check[
        "alfabetizado"
    ].isna().sum()
)

print(
    "Disponibilidade Gold:"
)

print(
    modeling_gold_check[
        "gold_historico_disponivel"
    ].value_counts()
)